In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../..')

from Model.Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Model.Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Model.Pro.DataPrep import Prep_Map, Output_Map
from Model.College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map
from Model.Constants import device, NUM_MODEL_VARIANTS

In [ ]:
is_hitter = True

In [ ]:
from Model.Constants import GetDataPrepBinaryFile

data_prep = Combined_Data_Prep(
    Prep_Map.base_prep_map, 
    Output_Map.base_output_map,
    C_Prep_Map.college_base_prep_map,
    C_Output_Map.college_output_map,
    save_name="../" + GetDataPrepBinaryFile(1))

data_prep = Combined_Data_Prep.Load_From_File("../" + GetDataPrepBinaryFile(1))

In [ ]:
io_list = data_prep.Generate_IO_Hitters(is_training=True) if is_hitter else \
    data_prep.Generate_IO_Pitchers(is_training=True)

In [ ]:
war_losses = []
stats_losses = []

stats_grad_scales = [0, 0.01, 0.02, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]

In [ ]:
from tqdm.auto import tqdm
from Model.Pro.Model.Player_Model import DEFAULT_HITTER_GRAD_SCALES, LOSS_IDX_STATS
from Model.Pro.Model.Player_Model import Recurrent_Model as Pro_Model
from Model.College.Model.College_Model import RNN_Model as College_Model
from Model.Constants import device
import gc
import torch
from Model.Combined.Model.Model_Train import TrainAndGraph

for stat_scale in tqdm(stats_grad_scales, desc="Stat Grad Scales"):
    war_loss = 0
    stats_loss = 0
    for var_idx in tqdm(range(NUM_MODEL_VARIANTS), desc="Model Variants", leave=False):
        # Load Data
        train_dataset, test_dataset = Create_Test_Train_Datasets(
            player_list=io_list, 
            is_hitter=is_hitter,
            train_idx=var_idx)
        
        # Get new model variant
        gs = list(DEFAULT_HITTER_GRAD_SCALES)
        gs[LOSS_IDX_STATS] = stat_scale

        pro_model = Pro_Model(
            input_size=data_prep.GetProIOSize(is_hitter),
            data_prep=data_prep.pro_data_prep,
            is_hitter=is_hitter,
            trunk_grad_scales=gs
        ).to(device)
        col_model = College_Model(
            input_size=data_prep.GetColIOSize(is_hitter),
            data_prep=data_prep.college_data_prep,
            is_hitter=is_hitter,
            output_init_state_size=pro_model.GetInitStateSize(),
        ).to(device)
        
        # Run Model
        train_results = TrainAndGraph(
            pro_network=pro_model,
            col_network=col_model,
            train_dataset=train_dataset,
            test_dataset=test_dataset,
            pro_model_name="../Models/test_pro",
            col_model_name="../Models/test_col",
            is_hitter=is_hitter,
            should_output=False,
            show_progress_bar=False
        )
        
        # Save Results
        war_loss += train_results.best_loss_war
        stats_loss += train_results.best_loss_stats
        
        # Force memory cleanup, since if the GC doesn't decide to cleanup 
        # before the next run can get OOM
        del train_dataset
        del test_dataset
        del pro_model
        del col_model
        gc.collect()
        torch.cuda.empty_cache()
        
    # Store Aggregate Results
    war_losses.append(war_loss)
    stats_losses.append(stats_loss)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(stats_grad_scales, war_losses)

ax.set_xscale("log")
ax.set_xlabel("Stats Grad Weight")
ax.set_ylabel(f"Sum of War Losses over {NUM_MODEL_VARIANTS} variants")
ax.set_title(f"Stats Grad Scale vs. WAR loss ({'hitters' if is_hitter else 'pitchers'})")
ax.grid(True, which="both", alpha=0.3)
plt.savefig("Graphs/WAR_vs_Stats_Grad_Scale.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(stats_grad_scales, stats_losses)

ax.set_xscale("log")
ax.set_xlabel("Stats Grad Weight")
ax.set_ylabel(f"Sum of Stats Losses over {NUM_MODEL_VARIANTS} variants")
ax.set_title(f"Stats Grad Scale vs. Stats loss ({'hitters' if is_hitter else 'pitchers'})")
ax.grid(True, which="both", alpha=0.3)
plt.savefig("Graphs/Stats_vs_Stats_Grad_Scale.png")
plt.show()